# 🏊 AquaGuard AI — Master Notebook: Two-Model Child Safety System
**BUS61104 · Taylor's University · Group 20 · April 2026**

---

### System overview
```
Camera frame
     ↓
Model 1 — YOLOv8m-seg  (Notebook 01)
     → Detects pool water polygon
     ↓
Model 2 — YOLOv9c  (Notebook 02 MERGED)
     → Classifies every person as CHILD or ADULT
     ↓
Alert Logic
     → Child centroid INSIDE pool polygon
     + NO adult detected ANYWHERE in frame
     + Confirmed for 5 consecutive frames
     → WhatsApp EMERGENCY alert → Parent
```

### Models used
| # | Model | Source notebook | Key metric |
|---|---|---|---|
| 1 | YOLOv8m-seg | `01_pool_segmentation.ipynb` | mAP50 = 95.12% |
| 2 | YOLOv9c | `02_child_adult_detection_MERGED.ipynb` | Child AP50 = 92.78% |

### What this notebook covers
| Section | What it does |
|---|---|
| 1 · Setup | Packages, imports, Drive |
| 2 · Load models | Load weights from Notebooks 01 and 02 |
| 3 · Pipeline | Define alert logic + annotation |
| 4 · Static test | Run on validation images from both datasets |
| 5 · Benchmark | Measure combined pipeline speed |
| 6 · Video deployment | Upload video → annotated output + accuracy report |
| 7 · Evaluation | Figures, metrics, system summary |

---


## 1 · Setup

In [ ]:
!pip install ultralytics roboflow opencv-python matplotlib seaborn pandas -q
!apt-get install -y ffmpeg -q
print('✓ Ready')

In [ ]:
import os, cv2, json, glob, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict, deque
from IPython.display import HTML, display
from base64 import b64encode
from ultralytics import YOLO

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
print('✓ Imports ready')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = Path('/content/drive/MyDrive/AquaGuard_AI')
RES       = DRIVE / 'results'
MASTER    = RES / 'master_2model'
MASTER.mkdir(parents=True, exist_ok=True)

OUTDIR = Path('/content/aquaguard_2model')
OUTDIR.mkdir(exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Results: {RES}')
print(f'  Master:  {MASTER}')

---
## 2 · Load Both Models

Loads weights trained in Notebooks 01 and 02.  
Uses `best_model.json` from each notebook to confirm which variant was selected.


In [ ]:
# ── Model paths — exact folder names from training ───────────────
WEIGHT_PATHS = {
    'seg'  : RES/'model1_pool_seg'/'training'/'yolov8m_seg'/'weights'/'best.pt',
    'child': RES/'model2_child_adult'/'training'/'yolov9c'/'weights'/'best.pt',
}

# Auto-search fallback if exact path missing
for key, path in list(WEIGHT_PATHS.items()):
    if not path.exists():
        folder = 'model1_pool_seg' if key=='seg' else 'model2_child_adult'
        found  = glob.glob(str(RES/folder/'training'/'**'/'best.pt'),
                           recursive=True)
        if found:
            # Prefer matching variant name
            target = 'yolov8m_seg' if key=='seg' else 'yolov9c'
            pref   = [w for w in found if target in Path(w).parts[-3].lower()]
            WEIGHT_PATHS[key] = Path(pref[0] if pref else found[0])
            print(f'  {key}: found → .../{WEIGHT_PATHS[key].parts[-3]}/...')

# Load
print('Loading models...')
seg_model   = YOLO(str(WEIGHT_PATHS['seg']))   if WEIGHT_PATHS['seg'].exists()   else None
child_model = YOLO(str(WEIGHT_PATHS['child'])) if WEIGHT_PATHS['child'].exists() else None

# Confirm from saved JSON
for key, folder in [('seg','model1_pool_seg'),('child','model2_child_adult')]:
    jp = RES/folder/'best_model.json'
    if jp.exists():
        info = json.load(open(jp))
        m50  = (info.get('metrics',{}).get('mAP50 (mask)')
                or info.get('metrics',{}).get('mAP50','—'))
        print(f'  ✓ {key}: {info["best_variant"]}  mAP50={m50}')
    else:
        print(f'  ✓ {key}: loaded (no JSON found)')

if seg_model and child_model:
    print('\n✓ Both models loaded')
else:
    print('\n⚠ Check Drive paths — run Notebooks 01 and 02 first')

In [ ]:
# Confirm class names for Model 2
# Merged notebook trains: adult=0, child=1  (elderly folded into adult)
jp2 = RES/'model2_child_adult'/'best_model.json'
if jp2.exists():
    info2 = json.load(open(jp2))
    CLASS_NAMES = info2.get('metrics',{}).get('class_names',['adult','child'])
else:
    CLASS_NAMES = ['adult','child']

ADULT_IDX = next((i for i,n in enumerate(CLASS_NAMES) if n=='adult'), 0)
CHILD_IDX = next((i for i,n in enumerate(CLASS_NAMES) if n=='child'), 1)

print(f'Class names:  {CLASS_NAMES}')
print(f'Adult index:  {ADULT_IDX}')
print(f'Child index:  {CHILD_IDX}')

---
## 3 · Pipeline Architecture

**Alert rule (all three conditions must be true):**
1. Pool polygon detected (Model 1)
2. At least one child centroid is **inside** the pool polygon
3. **No adult detected anywhere** in the frame
4. Condition holds for **5 consecutive frames** (reduces false positives)


In [ ]:
# ── Configuration ─────────────────────────────────────────────────
CONF_SEG   = 0.50   # pool segmentation confidence
CONF_CHILD = 0.40   # child/adult detection confidence

CONFIRM_FRAMES = 5  # consecutive frames before alert fires
ALERT_COOLDOWN = 30 # seconds between repeated alerts of the same type

# Colours for annotation (BGR)
C_POOL  = (0,  229, 255)   # teal  — pool zone
C_CHILD = (0,  165, 255)   # orange — child
C_ADULT = (178,145,  8)    # blue  — adult
C_ALERT = (0,   0,  220)   # red   — alert banner

print('Pipeline configuration:')
print(f'  Pool seg conf:    {CONF_SEG}')
print(f'  Child/adult conf: {CONF_CHILD}')
print(f'  Confirm frames:   {CONFIRM_FRAMES}')
print(f'  Alert cooldown:   {ALERT_COOLDOWN}s')

In [ ]:
# ── Pool zone helpers ─────────────────────────────────────────────
MANUAL_POLYGON = np.array([[100,80],[540,85],[550,380],[90,375]],
                           dtype=np.int32)

def get_pool_polygon(frame, seg_model, conf=0.5):
    """Run seg model → return pool polygon (or None)."""
    res = seg_model(frame, conf=conf, verbose=False)[0]
    if res.masks is not None:
        for mask, cls in zip(res.masks.xy, res.boxes.cls):
            if int(cls) == 0:
                return np.array(mask, dtype=np.int32)
    return None

def point_in_pool(cx, cy, polygon):
    """Returns True if point (cx,cy) is inside the pool polygon."""
    return cv2.pointPolygonTest(
        polygon, (float(cx), float(cy)), False) >= 0

def draw_pool(frame, polygon, alpha=0.15):
    """Draw semi-transparent pool fill + teal border."""
    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon], (0,100,60))
    cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
    cv2.polylines(frame, [polygon], True, C_POOL, 2)
    cv2.putText(frame, 'Pool Zone', (polygon[0][0], polygon[0][1]-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, C_POOL, 1)
    return frame

print('✓ Pool zone helpers ready')

In [ ]:
# ── Main pipeline function ────────────────────────────────────────
alert_counter = defaultdict(int)  # tracks consecutive alert frames
last_alert_t  = 0.0


def run_pipeline(frame, pool_poly=None, use_seg=True):
    """
    Run both models on one frame.
    Returns: (annotated_frame, alert_type, details_dict)
      alert_type: 'CHILD_ALONE' | None
    """
    global alert_counter
    annotated = frame.copy()
    h, w      = frame.shape[:2]
    details   = {'pool_found':False,'children_in_pool':0,'adult_visible':False}

    # ── MODEL 1: Pool segmentation ────────────────────────────────
    t1 = time.perf_counter()
    if use_seg and seg_model:
        detected = get_pool_polygon(frame, seg_model, CONF_SEG)
        if detected is not None:
            pool_poly = detected
    ms_seg = (time.perf_counter()-t1)*1000

    if pool_poly is None:
        cv2.putText(annotated,'No pool detected',(10,35),
                    cv2.FONT_HERSHEY_SIMPLEX,0.8,(100,100,100),2)
        return annotated, None, details

    draw_pool(annotated, pool_poly)
    details['pool_found'] = True

    # ── MODEL 2: Child / Adult detection ─────────────────────────
    t2 = time.perf_counter()
    det_res = child_model(frame, conf=CONF_CHILD, verbose=False)[0]
    ms_det  = (time.perf_counter()-t2)*1000
    ms_total= ms_seg + ms_det

    adult_anywhere   = False
    children_in_pool = []

    for box in det_res.boxes:
        cls  = int(box.cls)
        cf   = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cx, cy = (x1+x2)//2, y2  # foot point for pool test

        is_child = (cls == CHILD_IDX)
        color    = C_CHILD if is_child else C_ADULT
        label    = CLASS_NAMES[cls] if cls<len(CLASS_NAMES) else str(cls)

        # Track adults anywhere in frame
        if not is_child:
            adult_anywhere = True

        # Track children inside pool
        if is_child and point_in_pool(cx, cy, pool_poly):
            children_in_pool.append((x1,y1,x2,y2))
            # Red border for child IN pool
            cv2.rectangle(annotated,(x1,y1),(x2,y2),(0,0,220),3)
        else:
            cv2.rectangle(annotated,(x1,y1),(x2,y2),color,2)

        # Label with background
        lbl_txt = f'{label} {cf:.2f}'
        (lw,lh),_ = cv2.getTextSize(lbl_txt,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(annotated,(x1,y1-lh-8),(x1+lw+4,y1),color,-1)
        cv2.putText(annotated,lbl_txt,(x1+2,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0),1)

        # Foot-point dot for pool test visibility
        cv2.circle(annotated,(cx,cy),5,color,-1)

    details.update({'children_in_pool':len(children_in_pool),
                    'adult_visible':adult_anywhere,
                    'ms_seg':round(ms_seg,2),
                    'ms_det':round(ms_det,2),
                    'ms_total':round(ms_total,2)})

    # ── Alert logic ───────────────────────────────────────────────
    alert_type = None
    if children_in_pool and not adult_anywhere:
        alert_counter['child_alone'] += 1
        if alert_counter['child_alone'] >= CONFIRM_FRAMES:
            alert_type = 'CHILD_ALONE'
    else:
        alert_counter['child_alone'] = max(
            0, alert_counter['child_alone']-1)

    # Draw alert banner
    if alert_type:
        cv2.rectangle(annotated,(0,45),(annotated.shape[1],115),
                      (0,0,160),-1)
        cv2.rectangle(annotated,(0,45),(annotated.shape[1],115),
                      C_ALERT,3)
        cv2.putText(annotated,'⚠  CHILD ALONE IN POOL — ALERTING PARENT',
                    (annotated.shape[1]//2-320,82),
                    cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
    elif children_in_pool and not adult_anywhere:
        # Building toward alert
        cnt = alert_counter['child_alone']
        cv2.putText(annotated,
                    f'⚠ Child alone — confirming ({cnt}/{CONFIRM_FRAMES})',
                    (10,50),cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,165,255),2)

    return annotated, alert_type, details


print('✓ Pipeline function ready')
print('  Alert fires when: child IN pool + NO adult in frame '
      f'for {CONFIRM_FRAMES} consecutive frames')

---
## 4 · Static Image Test

Run the pipeline on validation images from both datasets to verify detection quality.


In [ ]:
# Download both validation sets
from roboflow import Roboflow
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print('✓ API key from Colab Secrets')
except Exception:
    API_KEY = 'PASTE_YOUR_KEY_HERE'

rf = Roboflow(api_key=API_KEY)
test_images = {'seg':[], 'child':[]}

# Pool seg validation images
try:
    ds1 = rf.workspace('ehsangooyahotmailcom-6xaas')\
            .project('pool-localisation').version(1).download('yolov8')
    imgs = list((Path(ds1.location)/'valid'/'images').glob('*.jpg'))
    test_images['seg'] = random.sample(imgs, min(12, len(imgs)))
    print(f'✓ Pool seg:   {len(test_images["seg"])} test images')
except Exception as e:
    print(f'  ✗ Pool seg: {e}')

# Child/Adult validation images (merged dataset)
# Check if merged dataset already exists on disk
merged_val = Path('/content/merged/valid/images')
if merged_val.exists():
    imgs = list(merged_val.glob('*.jpg')) + list(merged_val.glob('*.png'))
    test_images['child'] = random.sample(imgs, min(12, len(imgs)))
    print(f'✓ Child/adult: {len(test_images["child"])} images (from merged)')
else:
    try:
        ds2 = rf.workspace('kpz2').project('child-adult-elderly')\
                .version(2).download('yolov8')
        imgs = list((Path(ds2.location)/'valid'/'images').glob('*.jpg'))
        test_images['child'] = random.sample(imgs, min(12, len(imgs)))
        print(f'✓ Child/adult: {len(test_images["child"])} test images')
    except Exception as e:
        print(f'  ✗ Child/adult: {e}')

In [ ]:
# Test Model 1 alone — pool segmentation
samples = random.sample(test_images['seg'], min(6, len(test_images['seg'])))
fig, axes = plt.subplots(2,3,figsize=(16,9))
fig.suptitle('Model 1 — Pool Segmentation (YOLOv8m-seg)',
             fontsize=13,fontweight='bold')
for ax, ip in zip(axes.flatten(), samples):
    frame = cv2.imread(str(ip))
    if frame is None: ax.axis('off'); continue
    frame = cv2.resize(frame,(640,640))
    poly  = get_pool_polygon(frame, seg_model, CONF_SEG)
    out   = frame.copy()
    if poly is not None:
        draw_pool(out, poly)
        title_sfx='✓ pool detected'
    else:
        title_sfx='✗ no pool'
    ax.imshow(cv2.cvtColor(out,cv2.COLOR_BGR2RGB))
    ax.set_title(f'{Path(ip).name[:20]}\n{title_sfx}',fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(samples):]: ax.axis('off')
plt.tight_layout()
plt.savefig(str(MASTER/'test_seg_model.png'),dpi=120,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/test_seg_model.png')

In [ ]:
# Test Model 2 alone — child/adult detection
samples2 = random.sample(test_images['child'], min(6,len(test_images['child'])))
fig, axes = plt.subplots(2,3,figsize=(16,9))
fig.suptitle('Model 2 — Child/Adult Detection (YOLOv9c MERGED)',
             fontsize=13,fontweight='bold')
for ax, ip in zip(axes.flatten(), samples2):
    frame = cv2.imread(str(ip))
    if frame is None: ax.axis('off'); continue
    frame = cv2.resize(frame,(640,640))
    res   = child_model(frame,conf=CONF_CHILD,verbose=False)[0]
    out   = res.plot()
    ax.imshow(cv2.cvtColor(out,cv2.COLOR_BGR2RGB))
    n_c=sum(1 for b in res.boxes if int(b.cls)==CHILD_IDX)
    n_a=sum(1 for b in res.boxes if int(b.cls)==ADULT_IDX)
    ax.set_title(f'{Path(ip).name[:20]}\n{n_c}C / {n_a}A',fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(samples2):]: ax.axis('off')
plt.tight_layout()
plt.savefig(str(MASTER/'test_child_model.png'),dpi=120,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/test_child_model.png')

In [ ]:
# Test full 2-model pipeline on pool images with manual polygon
# (best test: pool image WITH people simulated)
FULL_TEST_POLY = np.array([[80,60],[560,65],[570,420],[75,415]],dtype=np.int32)

all_test = test_images['seg'] + test_images['child']
samples3  = random.sample(all_test, min(6,len(all_test)))

fig, axes = plt.subplots(2,3,figsize=(16,10))
fig.suptitle('Full 2-Model Pipeline — Pool Seg + Child/Adult Detection\n'
             'Teal=pool zone  Orange=child  Blue=adult  Red border=child IN pool',
             fontsize=11,fontweight='bold')

for ax, ip in zip(axes.flatten(), samples3):
    frame = cv2.imread(str(ip))
    if frame is None: ax.axis('off'); continue
    frame = cv2.resize(frame,(640,640))
    alert_counter.clear()
    out, alert_type, det = run_pipeline(frame, pool_poly=FULL_TEST_POLY,
                                         use_seg=True)
    n_c = det['children_in_pool']
    has_adult = det['adult_visible']
    status = '🚨 ALERT' if alert_type else (
             f'{n_c}C in pool — adult:{"+" if has_adult else "NO"}'
             if n_c>0 else 'safe')
    ax.imshow(cv2.cvtColor(out,cv2.COLOR_BGR2RGB))
    ax.set_title(f'{Path(ip).name[:18]}\n{status}',fontsize=8)
    ax.axis('off')

for ax in axes.flatten()[len(samples3):]: ax.axis('off')
plt.tight_layout()
plt.savefig(str(MASTER/'test_pipeline_combined.png'),dpi=120,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/test_pipeline_combined.png')

---
## 5 · Pipeline Speed Benchmark

In [ ]:
N_BENCH = 50
dummy   = np.zeros((640,640,3),dtype=np.uint8)
BENCH_POLY = np.array([[100,80],[540,85],[550,380],[90,375]],dtype=np.int32)

# Warm-up
run_pipeline(dummy, BENCH_POLY, use_seg=False)

# Mode A: hardcoded polygon (production mode)
times_a = {'seg':[],'det':[],'total':[]}
for _ in range(N_BENCH):
    _,_,d = run_pipeline(dummy, BENCH_POLY, use_seg=False)
    times_a['seg'].append(d.get('ms_seg',0))
    times_a['det'].append(d.get('ms_det',0))
    times_a['total'].append(d.get('ms_total',0))

# Mode B: live seg per frame
times_b = {'seg':[],'det':[],'total':[]}
for _ in range(N_BENCH):
    _,_,d = run_pipeline(dummy, use_seg=True)
    times_b['seg'].append(d.get('ms_seg',0))
    times_b['det'].append(d.get('ms_det',0))
    times_b['total'].append(d.get('ms_total',0))

ta = np.mean(times_a['total']); fps_a = round(1000/max(ta,0.1),1)
tb = np.mean(times_b['total']); fps_b = round(1000/max(tb,0.1),1)

print('='*52)
print('  PIPELINE BENCHMARK — 2-MODEL SYSTEM')
print('='*52)
print(f'  {'Stage':<22} {'Mode A':>10} {'Mode B':>10}')
print(f'  {"Model 1 Pool Seg":<22} '
      f'{np.mean(times_a["seg"]):>9.1f}ms '
      f'{np.mean(times_b["seg"]):>9.1f}ms')
print(f'  {"Model 2 Child/Adult":<22} '
      f'{np.mean(times_a["det"]):>9.1f}ms '
      f'{np.mean(times_b["det"]):>9.1f}ms')
print(f'  {"TOTAL":<22} {ta:>9.1f}ms {tb:>9.1f}ms')
print(f'  {"FPS":<22} {fps_a:>10} {fps_b:>10}')
print('='*52)
print('  Mode A = hardcoded polygon (deployment)  ← USE THIS')
print('  Mode B = seg model every frame')

In [ ]:
# Latency breakdown figure
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('2-Model Pipeline — Latency Analysis',fontsize=13,fontweight='bold')

s_ms   = [np.mean(times_a['seg']), np.mean(times_a['det'])]
s_lbl  = ['Model 1\nPool Seg','Model 2\nChild/Adult']
s_clrs = ['#00E5FF','#F97316']

bars = axes[0].barh(s_lbl, s_ms, color=s_clrs, edgecolor='none', height=0.4)
axes[0].set_xlabel('ms')
axes[0].set_title(f'Mode A (hardcoded zone)\n{ta:.1f}ms total → {fps_a} FPS')
for bar,ms in zip(bars,s_ms):
    axes[0].text(max(ms,0.05)+0.1, bar.get_y()+bar.get_height()/2,
                 f'{ms:.1f}ms', va='center', fontsize=11, fontweight='bold')

bars2 = axes[1].barh(s_lbl,
                     [np.mean(times_b['seg']),np.mean(times_b['det'])],
                     color=s_clrs,edgecolor='none',height=0.4)
axes[1].set_xlabel('ms')
axes[1].set_title(f'Mode B (live seg every frame)\n{tb:.1f}ms total → {fps_b} FPS')
for bar,ms in zip(bars2,[np.mean(times_b['seg']),np.mean(times_b['det'])]):
    axes[1].text(ms+0.1,bar.get_y()+bar.get_height()/2,
                 f'{ms:.1f}ms',va='center',fontsize=11,fontweight='bold')

pcts = [max(v,0.001) for v in s_ms]
axes[2].pie(pcts,labels=s_lbl,colors=s_clrs,autopct='%1.1f%%',
            startangle=90,wedgeprops={'edgecolor':'#0F172A','linewidth':2})
axes[2].set_title(f'Stage contribution (Mode A)\n{ta:.1f}ms total')

plt.tight_layout()
plt.savefig(str(MASTER/'pipeline_latency.png'),dpi=120,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/pipeline_latency.png')

---
## 6 · Video Deployment

Upload any video → full pipeline runs on every frame →  
annotated output video + accuracy/alert report.


### Option A — YouTube

In [ ]:
!pip install yt-dlp -q 2>/dev/null
YOUTUBE_URL = 'https://www.youtube.com/watch?v=PASTE_URL_HERE'
!yt-dlp -f 'best[height<=480][ext=mp4]/best[height<=480]' \
        --output '/content/test_video.%(ext)s' --no-playlist '{YOUTUBE_URL}'
import glob as _g
vids = _g.glob('/content/test_video.*')
if vids:
    INPUT_VIDEO = vids[0]
    cap_=cv2.VideoCapture(INPUT_VIDEO)
    print(f'✓ {INPUT_VIDEO}  '
          f'{int(cap_.get(3))}×{int(cap_.get(4))}  '
          f'{cap_.get(5):.1f}fps  {int(cap_.get(7))} frames')
    cap_.release()

### Option B — Upload

In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    INPUT_VIDEO = '/content/' + list(uploaded.keys())[0]
    cap_=cv2.VideoCapture(INPUT_VIDEO)
    print(f'✓ {INPUT_VIDEO}  '
          f'{int(cap_.get(3))}×{int(cap_.get(4))}  '
          f'{cap_.get(5):.1f}fps  {int(cap_.get(7))} frames')
    cap_.release()

In [ ]:
# ── Process video ────────────────────────────────────────────────
OUTPUT_RAW  = str(OUTDIR/'output_raw.mp4')
OUTPUT_H264 = str(OUTDIR/'output_final.mp4')
MAX_FRAMES  = None
SKIP_FRAMES = 1

cap     = cv2.VideoCapture(INPUT_VIDEO)
fps_in  = cap.get(cv2.CAP_PROP_FPS) or 25
w_in    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_in    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
max_f   = min(n_total, MAX_FRAMES) if MAX_FRAMES else n_total

writer = cv2.VideoWriter(OUTPUT_RAW,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps_in/SKIP_FRAMES, (w_in,h_in))

print(f'Input:  {w_in}×{h_in}  {fps_in:.1f}fps  {n_total} frames')
print(f'Output: {OUTPUT_RAW}')
print('Running...\n')

# State
alert_counter.clear()
pool_poly_live = MANUAL_POLYGON
frame_stats   = []
alert_log     = []
last_alert_t  = 0.0
frame_times   = []
n_alerts      = 0
frame_idx     = 0; proc_idx = 0
t_start       = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_idx >= max_f: break
    frame_idx += 1
    if frame_idx % SKIP_FRAMES != 0: continue
    proc_idx += 1

    t0 = time.perf_counter()

    # Run pipeline (Mode A — hardcoded polygon for speed)
    out, alert_type, det = run_pipeline(
        cv2.resize(frame,(640,640)),
        pool_poly=pool_poly_live, use_seg=False)
    out = cv2.resize(out, (w_in,h_in))

    inf_ms = (time.perf_counter()-t0)*1000
    frame_times.append(inf_ms)

    frame_stats.append({
        'frame'           : frame_idx,
        'children_in_pool': det['children_in_pool'],
        'adult_visible'   : det['adult_visible'],
        'alert'           : alert_type is not None,
    })

    # Log alert with cooldown
    if alert_type and (time.time()-last_alert_t) > ALERT_COOLDOWN:
        ts_str = str(timedelta(seconds=int(frame_idx/fps_in)))
        alert_log.append({'type':alert_type,'time':ts_str,
                           'frame':frame_idx})
        last_alert_t = time.time()
        n_alerts += 1

    # HUD
    disp_fps = 1000/max(np.mean(frame_times[-30:]),0.1)
    ts_str   = str(timedelta(seconds=int(frame_idx/fps_in)))
    cv2.rectangle(out,(0,0),(w_in,50),(10,10,10),-1)
    cv2.putText(out,'AquaGuard AI',(10,33),
                cv2.FONT_HERSHEY_SIMPLEX,0.85,(0,229,255),2)
    cv2.putText(out,f'{ts_str}  {disp_fps:.1f}FPS',
                (w_in-210,33),cv2.FONT_HERSHEY_SIMPLEX,0.55,(180,180,180),1)
    cv2.putText(out,f'Alerts:{n_alerts}',
                (w_in//2-60,33),cv2.FONT_HERSHEY_SIMPLEX,0.65,
                (0,0,220) if n_alerts>0 else (200,200,200),1)

    # Status bar
    n_c=det['children_in_pool']; has_a=det['adult_visible']
    cv2.rectangle(out,(0,h_in-28),(w_in,h_in),(10,10,10),-1)
    st_col = (0,0,220) if n_c>0 and not has_a else \
             (0,229,255) if n_c>0 else (129,185,16)
    st_txt = ('⚠ CHILD ALONE' if n_c>0 and not has_a else
              f'{n_c} child in pool (supervised)' if n_c>0 else
              'Pool safe')
    cv2.putText(out,f'Status: {st_txt}',(8,h_in-10),
                cv2.FONT_HERSHEY_SIMPLEX,0.45,st_col,1)

    writer.write(out)

    if proc_idx % 60 == 0:
        pct = frame_idx/max_f*100
        eta = (time.time()-t_start)/max(frame_idx,1)*(max_f-frame_idx)
        print(f'  [{pct:5.1f}%] {frame_idx}/{max_f}  '
              f'{disp_fps:.1f}fps  ETA:{eta:.0f}s  alerts:{n_alerts}')

cap.release(); writer.release()
print(f'\n✓ Done in {time.time()-t_start:.1f}s  |  {proc_idx} frames')
print(f'  Alerts fired: {n_alerts}')

# Convert to H.264
!ffmpeg -i '{OUTPUT_RAW}' -vcodec libx264 -crf 22 -preset fast \
        '{OUTPUT_H264}' -y -loglevel quiet
print(f'✓ H.264 video: {OUTPUT_H264}')

In [ ]:
# Display output video inline
def show_video(path, width=800):
    data = b64encode(open(path,'rb').read()).decode()
    return HTML(f"""
    <div style='text-align:center;background:#0F172A;padding:12px;border-radius:8px;'>
      <h3 style='color:#00E5FF;font-family:sans-serif;'>
        AquaGuard AI — 2-Model System Output</h3>
      <video width={width} controls autoplay muted loop
             style='border:2px solid #00E5FF;border-radius:4px;'>
        <source src='data:video/mp4;base64,{data}' type='video/mp4'>
      </video>
      <p style='color:#64748B;font-size:12px;margin-top:6px;'>
        Teal = pool zone &nbsp;|&nbsp; Orange = child &nbsp;|
        &nbsp; Blue = adult &nbsp;|&nbsp; Red border = child IN pool
      </p>
    </div>""")

display(show_video(OUTPUT_H264))

---
## 7 · Evaluation

In [ ]:
# Build evaluation figures
fig, axes = plt.subplots(2,2,figsize=(16,11))
fig.suptitle('AquaGuard AI — 2-Model System Evaluation',
             fontsize=14,fontweight='bold')

fs_df = pd.DataFrame(frame_stats)
x     = fs_df['frame'].values
roll  = 20

# Chart 1: Children in pool over time
c_vals = fs_df['children_in_pool'].values
c_sm   = pd.Series(c_vals.astype(float)).rolling(roll,min_periods=1).mean()
axes[0][0].fill_between(x, c_sm, alpha=0.3, color='#F97316')
axes[0][0].plot(x, c_sm, color='#F97316', linewidth=1.8,
                label='Children in pool')
axes[0][0].axhline(1, color='gray', linestyle='--', alpha=0.4)
axes[0][0].set_xlabel('Frame'); axes[0][0].set_ylabel('Count')
axes[0][0].set_title('Children Detected Inside Pool (rolling avg)')
axes[0][0].legend()

# Chart 2: Alert timeline
alert_frames_idx = [fs['frame'] for fs in frame_stats if fs['alert']]
axes[0][1].scatter(alert_frames_idx, [1]*len(alert_frames_idx),
                   color='#EF4444', s=15, alpha=0.7)
axes[0][1].set_xlim(0, max_f)
axes[0][1].set_ylim(0,2); axes[0][1].set_yticks([])
axes[0][1].set_xlabel('Frame')
pct_alert = len(alert_frames_idx)/max(proc_idx,1)*100
axes[0][1].set_title(f'Alert Timeline (Child Alone in Pool)\n'
                     f'{len(alert_frames_idx)} alert frames '
                     f'({pct_alert:.1f}%) | {n_alerts} unique alerts fired')

# Chart 3: Per-frame pipeline speed
axes[1][0].plot(range(len(frame_times)), frame_times,
                color='#00E5FF', linewidth=0.8, alpha=0.7)
axes[1][0].axhline(np.mean(frame_times), color='#F97316',
                   linewidth=2, linestyle='--',
                   label=f'Mean {np.mean(frame_times):.1f}ms')
axes[1][0].set_xlabel('Processed frame')
axes[1][0].set_ylabel('ms')
axes[1][0].set_title(f'Pipeline Speed per Frame\n'
                     f'Mean={np.mean(frame_times):.1f}ms → '
                     f'{1000/np.mean(frame_times):.1f} FPS')
axes[1][0].legend()

# Chart 4: Summary metrics table
axes[1][1].axis('off')
n_child_frames = int((fs_df['children_in_pool']>0).sum())
n_sup_frames   = int(((fs_df['children_in_pool']>0) & fs_df['adult_visible']).sum())
n_alone_frames = int(((fs_df['children_in_pool']>0) & ~fs_df['adult_visible']).sum())
rows_ = [
    ['Metric','Value'],
    ['Frames processed',      f'{proc_idx}'],
    ['Pool seg model',         'YOLOv8m-seg  mAP50=95.12%'],
    ['Child/adult model',      'YOLOv9c (MERGED dataset)'],
    ['Child class AP50',       '92.78% (from NB02)'],
    ['Frames: child in pool',  f'{n_child_frames} ({n_child_frames/max(proc_idx,1)*100:.1f}%)'],
    ['Frames: supervised',     f'{n_sup_frames}'],
    ['Frames: child ALONE',    f'{n_alone_frames} ({n_alone_frames/max(proc_idx,1)*100:.1f}%)'],
    ['Alerts fired',           f'{n_alerts}'],
    ['Avg pipeline speed',     f'{np.mean(frame_times):.1f}ms  ({1000/np.mean(frame_times):.1f} FPS)'],
    ['Confirm frames setting', f'{CONFIRM_FRAMES}'],
]
tbl = axes[1][1].table(cellText=rows_[1:], colLabels=rows_[0],
                        loc='center', cellLoc='left')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,1.55)
for (r,c),cell in tbl.get_celld().items():
    cell.set_facecolor('#1E293B' if r>0 else '#334155')
    cell.set_edgecolor('#475569')
    cell.set_text_props(color='white')
axes[1][1].set_title('System Summary', fontsize=11)

plt.tight_layout()
plt.savefig(str(MASTER/'system_evaluation.png'),dpi=130,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/system_evaluation.png')

In [ ]:
# Sample frames from output video
cap2 = cv2.VideoCapture(OUTPUT_H264)
nf   = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
idxs = [int(nf*p) for p in [0.05,0.2,0.4,0.6,0.8,0.95]]
fig, axes = plt.subplots(2,3,figsize=(18,10))
fig.suptitle('Sample Frames from Processed Video',fontsize=12,fontweight='bold')
for ax,idx in zip(axes.flatten(),idxs):
    cap2.set(cv2.CAP_PROP_POS_FRAMES,idx)
    ret,fr=cap2.read()
    if ret:
        ax.imshow(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
        ax.set_title(str(timedelta(seconds=int(idx/fps_in))),fontsize=9)
    ax.axis('off')
cap2.release()
plt.tight_layout()
plt.savefig(str(MASTER/'sample_frames.png'),dpi=120,bbox_inches='tight')
plt.show()
print('✓ Saved → master_2model/sample_frames.png')

In [ ]:
# Save system summary JSON
summary = {
    'system'        : 'AquaGuard AI — 2-Model Child Safety',
    'models'        : {
        'model1': {'name':'YOLOv8m-seg','task':'Pool segmentation',
                   'mAP50':0.9512,'notebook':'01_pool_segmentation'},
        'model2': {'name':'YOLOv9c','task':'Child/Adult detection',
                   'child_ap50':0.9278,'notebook':'02_MERGED'},
    },
    'pipeline'      : {'total_ms':round(float(np.mean(frame_times)),2),
                       'fps':round(1000/float(np.mean(frame_times)),1)},
    'alert_rule'    : 'child IN pool + NO adult in frame '
                      f'for {CONFIRM_FRAMES} consecutive frames',
    'alert_cooldown': ALERT_COOLDOWN,
    'video_results' : {
        'frames_processed'   : proc_idx,
        'child_alone_frames' : n_alone_frames,
        'alerts_fired'       : n_alerts,
        'alert_log'          : alert_log,
    },
    'generated_at'  : datetime.now().isoformat(),
}
json.dump(summary,
          open(str(MASTER/'system_summary.json'),'w'), indent=2)
print('✓ Saved → master_2model/system_summary.json')

# Download output files
from google.colab import files
files.download(OUTPUT_H264)
files.download(str(MASTER/'system_evaluation.png'))

print('\n'+'═'*52)
print('  MASTER NOTEBOOK COMPLETE — 2-MODEL SYSTEM')
print('═'*52)
print(f'  Pool seg:     YOLOv8m-seg   mAP50 = 95.12%')
print(f'  Child/adult:  YOLOv9c       Child AP50 = 92.78%')
print(f'  Pipeline:     {np.mean(frame_times):.1f}ms → '
      f'{1000/np.mean(frame_times):.1f} FPS (GPU)')
print(f'  Alerts fired: {n_alerts}')
print('═'*52)
print('\n  Ready for Raspberry Pi 5 + Hailo deployment')